# Using OpenAI models with Strands Agents via LiteLLM

## Overview

Strands Agents is an SDK that takes a model-driven approach to building and running AI agents in just a few lines of code. Strands supports multiple providers and models hosted anywhere.

[LiteLLM](https://docs.litellm.ai/docs/) is a unified interface for many LLM providers that lets you interact with models from Amazon, Anthropic, OpenAI, Azure OpenAI, and others through a single API. The Strands Agents SDK implements a LiteLLM provider, so you can run agents against any model LiteLLM supports.

In this example we use `gpt-4.1-mini` hosted on Microsoft Azure OpenAI as the underlying model for a Strands agent, with a simple use case: a `current_time` and a `current_weather` tool.

## Tutorial Details

| Information      | Details                                              |
|:-----------------|:-----------------------------------------------------|
| Agent structure  | Single agent                                         |
| Model provider   | LiteLLM (`LiteLLMModel`)                              |
| Model            | `gpt-4.1-mini` on Azure OpenAI (`azure/gpt-4.1-mini`) |
| Custom tools     | current_time, current_weather                        |
| Strands features | Reaching a hosted model through the LiteLLM provider |

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## What you'll learn
* Configure a hosted model with the `LiteLLMModel` provider
* Point LiteLLM at an Azure OpenAI deployment with environment variables
* Give the agent custom tools and inspect its messages and metrics

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* Azure Account
* gpt-4.1-mini access

Let's now install the requirement packages for our Strands Agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [ ]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.litellm import LiteLLMModel

### Setting up Azure keys

Let's now setup the Azure API Keys

In [ ]:
os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

### Setting up custom tools

Let's now setup two dummy tools to test our agent

In [ ]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)

    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Please replace with actual weather API call.
    return "sunny"

### Defining the agent's underlying model

Next let's define our agent's underlying model using LiteLLM. We set the `model_id` to `azure/gpt-4.1-mini`. With LiteLLM's `azure/<name>` format, the part after `azure/` is your Azure OpenAI *deployment name*, so update it to match the deployment you created in Azure.

In [ ]:
model = "azure/gpt-4.1-mini"
litellm_model = LiteLLMModel(
    model_id=model, params={"max_tokens": 32000, "temperature": 0.7}
)

### Defining Agent

Now that we have all the required information available, let's define our agent

In [ ]:
system_prompt = "You are a simple agent that can tell the time and the weather"
agent = Agent(
    model=litellm_model,
    system_prompt=system_prompt,
    tools=[current_time, current_weather],
)

### Testing agent

Let's now invoke the agent to test it

In [ ]:
results = agent("What time is it in Seattle? And how is the weather?")

#### Analysing the agent's results

Nice! We've invoked our agent for the first time! Let's now explore the results object. First thing we can see is the messages being exchanged by the agent in the agent's object

In [ ]:
agent.messages

Next we can take a look at the usage of our agent for the last query by analysing the result `metrics`

In [ ]:
results.metrics

## Summary

In this notebook you learned how to use the LiteLLM provider to run a Strands agent against an Azure OpenAI model, gave the agent custom tools, and inspected its messages and metrics. Next, let's use OpenAI models hosted on Amazon Bedrock through the Responses API.